In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
from default_risk.scripts.auxiliar_eda_function import check_invariant
import default_risk.config as cfg
import dtale
import dtale.global_state as dtale_global
import logging
import gc

installments_payment_df = pd.read_csv(cfg.INSTALLMENTS_PAYMENTS)

column_order_reference="DAYS_INSTALMENT"

data_frame_size=len(installments_payment_df)

installments_payment_df.sort_values(["SK_ID_PREV",column_order_reference,"DAYS_ENTRY_PAYMENT"],inplace=True)

log = logging.getLogger('werkzeug')

dtale_global.cleanup()
gc.collect()

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)


#aux function to avoid repeated code but keep idempotence between celds
def analyze_repeated_instalments(extra_mask,case_description) :
    next_installment_number = installments_payment_df.groupby("SK_ID_PREV")["NUM_INSTALMENT_NUMBER"].shift(-1)
    rows_to_analyze = installments_payment_df[
    (installments_payment_df ["NUM_INSTALMENT_NUMBER"] == next_installment_number) 
    &
    (extra_mask)]
    print("we have " + str(len(rows_to_analyze)) + " " + case_description + "\n")
    #display(rows_to_analyze.head(50))
    return rows_to_analyze

def get_full_sorted_serie_rows(rows : pd.DataFrame) -> pd.DataFrame:
   return recreate_and_sort_series_given_rows(rows,installments_payment_df, "SK_ID_PREV",column_order_reference)

def get_full_sorted_serie_ids(ids : list) -> pd.DataFrame:
    return recreate_and_sort_the_serie_given_ids(ids,installments_payment_df, "SK_ID_PREV" ,column_order_reference)
    




Invariants:

1- the column DAYS_INSTALMENT is left capped in -2922 (100%) #4

2- the values in column DAYS_ENTRY_PAYMENT < -2952 (99.99%) #4

3- If the payment is complete ("AMT_INSTALMENT" == "AMT_PAYMENT") and the version keeps the same between records (NUM_INSTALMENT_VERSION don't vary), the next row will not repeat the instalment number (NUM_INSTALMENT_VERSION behave strictly increasing) (100%) #13, third analysis

Soft constraints: 

1- The series starts in NUM_INSTALMENT_NUMBER = 1 or with a date older than -2890 in DAYS_INSTALMENT. Meaning that or we have register from the first installment or the first part
of the loan are beyond the time horizon to register data in this table. We have in train 997752 ids of previous contracts (temporal series) and just 253 series don't meet this rule.
(99.98%) #5
anomalies: incomplete series. 


Decisions summary: 

1- Missing values: We will divide the missing values  at AMT_PAYMENT -  DAYS_ENTRY_PAYMENT in 3 diferent cases at the moment of taking metrics.

    a- Dead tails: Where the values miss and stay as nan in the remaining rows of the serie and show now relevant change in other variable. This suggest administrative padding. 
    so we will ignore dead tails to the compute of agg metrics but keeping their length as a feature.

    b- useless anomalies: Change the instalment counter (NUM_INSTALMENT_NUMBER jump over 100) and have nans that are not cleary correlated with nothing relevant (like underpayment or any type of irregularities), so we are dropping this rows in the silver layer.
    
    c- remaining cases: Series where a missing value show ups and keep going normaly after all. Being less than 50 rows and the observed behavior we decide to fill this missings values with Forward-fill imputation.

2- Incomplete series: There is series that's does not start in NUM_INSTALMENT_NUMBER = 1 and they oldest date is < -2890 

3- Outliers handeling: For every value in DAYS_ENTRY_PAYMENT that don't fullfill the invariant #2 we will cap the value in DAYS_INSTALMENT + 30. In that way, if in the future we want to capture metrics of days of payments, that values will not destroy the proportion. 

4- Repeated instalments: We will divide the repeated instalment number in different cases: 
    a- If "AMT_PAYMENT"  < "AMT_INSTALMENT", we will count it as "repetition for underpayment"
    b- If "AMT_PAYMENT"  ==  "AMT_INSTALMENT" but the NUM_INSTALMENT_VERSION vary, we will considering reptition for reschedule 
    c- If "AMT_PAYMENT" == 0 and the row before acomplish the payment we will compute it as "repetition for payment in advance"

5- out of schedule tracking: We will count the rows with NUM_INSTALMENT_NUMBER > 100 and different NUM_INSTALMENT_VERSION that the first row of the sequence as "instalments_outside_schedule"

In [ ]:
#1
#files for the data dictionary
create_files_nulls_per_colmun(installments_payment_df,"installments_payment")

In [9]:
#2
#runing screening script
eda_per_table_printing_results(installments_payment_df,schema,"installments_payments",False)

--------------------------------------
SK_ID_PREV
basic_data


,cardinality,mode
0,997752,[2360056]


frequency


,CATEGORY,COUNT,SEGMENT
0,2360056,293,top
1,2592574,279,top
2,1017477,248,top
3,1449382,243,top
4,1746731,236,top
5,1690678,223,top
6,2709164,222,top
7,1383111,220,top
8,1152155,219,top
9,2543266,216,top


--------------------------------------
SK_ID_CURR
basic_data


...

,min,max,mean,trim_mean,median,standard_deviation,standard_error,coefficient_of_variation
0,0.0,3771487.845,17238.22325,9844.20088,8125.515,54735.783981,14.840962,3.175257


missings_metrics


,nulls_amount,nulls_porcentaje,non_null_amount,non_null_porcentaje
0,2905,0.021352,13602496,99.978648


distribution_metrics


,skew,p90,p99,ratio_p99_p50,ratio_p99_p90,ratio_max_value_p99
0,14.951925,31179.915,178003.95975,21.906791,5.70893,21.187663


In [ ]:
#3
prev_id= installments_payment_df["SK_ID_PREV"].unique()
series_to_show= prev_id[:100]
dtale.show(get_full_sorted_serie_ids(series_to_show),ignore_duplicate=True)


In [ ]:
#4
check_invariant(installments_payment_df["DAYS_INSTALMENT"] < -2922,"cases where have a older date than -2922 days", data_frame_size)

check_invariant(installments_payment_df["DAYS_ENTRY_PAYMENT"] < -2952,"cases where have a older date than -2922 days", data_frame_size)

In [ ]:
dtale.show(get_full_sorted_serie_rows(installments_payment_df[installments_payment_df["DAYS_ENTRY_PAYMENT"] < -3100]),ignore_duplicate=True)

In [ ]:
#5

grouped = installments_payment_df.groupby("SK_ID_PREV")

print(installments_payment_df["SK_ID_PREV"].nunique())

min_installment = grouped["NUM_INSTALMENT_NUMBER"].min()
min_days = grouped["DAYS_INSTALMENT"].min()

anomaly_condition = ( (min_installment > 1) & (min_days > -2890) )

anomaly_ids = anomaly_condition.index[anomaly_condition]

anomaly_rows = installments_payment_df[installments_payment_df["SK_ID_PREV"].isin(anomaly_ids)]

print(anomaly_rows["SK_ID_PREV"].nunique())

instance= dtale.show(get_full_sorted_serie_rows(anomaly_rows))
display(instance)

In [ ]:
#6
#in order to undestand the nature behind the missing values of DAYS_ENTRY_PAYMENT
rows_with_nulls=installments_payment_df[installments_payment_df["DAYS_ENTRY_PAYMENT"].isna()]
dtale.show(get_full_sorted_serie_rows(rows_with_nulls))

#in all the visualizated cases the tendence of the missing values is to show ups at the end of the secuence, like padding, so a "dead tail" definition is needed.
#Also, in the contract of SK_ID_PREV= 1004174 we can observe a jump from 8 to 101 in "NUM_INSTALMENT_NUMBER".
#This suggest that could exist errors in the counter or a sentinel / special value (101 show ups in more rows).

In [ ]:
#7
#now, let's validate the hipotesis of missing values in DAYS_ENTRY_PAYMENT y AMT_PAYMENT are dead tails. For that we will analize if exists cases
#that once the serie have a missing value in these columns, can exist a row with a value different a nan, of if once hit nan, it's always nan without relevant changes.
installments_payment_df["NEXT_VALUE_PAYMENT"] = installments_payment_df.groupby("SK_ID_PREV")["DAYS_ENTRY_PAYMENT"].shift(-1)
not_a_deadtail_mask= (installments_payment_df["DAYS_ENTRY_PAYMENT"].isna() ) & ( installments_payment_df["NEXT_VALUE_PAYMENT"].notna())
print(not_a_deadtail_mask.sum())
potencial_incosistencies_rows= installments_payment_df[not_a_deadtail_mask]
dtale.show(get_full_sorted_serie_rows(potencial_incosistencies_rows),ignore_duplicate=True)
#When we exclude dead tail cases the remaining observation of missing values in DAYS_ENTRY_PAYMENT are less than 50. 
#Also these series exhib an anormal behaivor. Seems like have another schedule with their own counter, using the prefix "100" (101,102,103...) in NUM_INSTALLMENT_NUMBER and a different NUM_INSTALLMEMT_VERSION.



In [ ]:
#8
#now let's analize the behaivor of the rows when the installment number jump to 100.
groups_sizes= installments_payment_df.groupby("SK_ID_PREV")["SK_ID_PREV"].transform("size")
series_bellow_hundred_rows= installments_payment_df[groups_sizes < 100]
rows_with_notation= series_bellow_hundred_rows[series_bellow_hundred_rows["NUM_INSTALMENT_NUMBER"] > 100]
dtale.show(get_full_sorted_serie_rows(rows_with_notation),ignore_duplicate=True) 
#This installments with diferent numeration (100 prefix) and also have a diferent (NUM_INSTALMENT_VERSION) appears to have a high correlation with repeated "NUM_INSTALMENT_NUMBER"
#and how this usually this represent a underpayment in that month we come with the hipotesis of, this specials insatllments could be extra fees for that.


In [ ]:
#9
grouped_per_contracts= installments_payment_df.groupby("SK_ID_PREV")
contracts_with_no_repeated_installments= grouped_per_contracts["NUM_INSTALMENT_NUMBER"].nunique() == grouped_per_contracts["NUM_INSTALMENT_NUMBER"].size()
ids_contracts_to_analize= contracts_with_no_repeated_installments.index[contracts_with_no_repeated_installments]
series_with_repetead_installments= installments_payment_df[installments_payment_df["SK_ID_PREV"].isin(ids_contracts_to_analize)]
groups_sizes= series_with_repetead_installments.groupby("SK_ID_PREV").transform("size")
series_bellow_hundred_rows= series_with_repetead_installments[(groups_sizes < 80)]
rows_with_notation= series_bellow_hundred_rows[series_bellow_hundred_rows["NUM_INSTALMENT_NUMBER"] > 100]
print(len(rows_with_notation))
dtale.show(get_full_sorted_serie_rows(rows_with_notation),ignore_duplicate=True) 

In [ ]:

#10
rows= installments_payment_df[(installments_payment_df["DAYS_INSTALMENT"] > installments_payment_df["DAYS_ENTRY_PAYMENT"])]
dtale.show(get_full_sorted_serie_rows(rows),ignore_duplicate=True)

In [ ]:
#11
grouped_per_contracts= installments_payment_df.groupby("SK_ID_PREV")
contracts_with_no_repeated_installments= grouped_per_contracts["NUM_INSTALMENT_NUMBER"].nunique() == grouped_per_contracts["NUM_INSTALMENT_NUMBER"].size()
ids_contracts_to_analize= contracts_with_no_repeated_installments.index[~contracts_with_no_repeated_installments]
series_with_repetead_installments= installments_payment_df[installments_payment_df["SK_ID_PREV"].isin(ids_contracts_to_analize)]
dtale.show(series_with_repetead_installments,ignore_duplicate=True)

In [ ]:
#12
#now, before end the EDA of this table we want to understand the nature behind of the repeated number of installments
rows_that_repeat_number= analyze_repeated_instalments(True,"of cases where the installment number are repeated")
dtale.show(get_full_sorted_serie_rows(rows_that_repeat_number))
#this show how repeated installments mainly represent underpayment / partial payment of a installment. So, let's check if could be generated for other reason.

In [7]:
#13
"""In order to show the cases where the instalment number tends to repeat, we create the cases of analysis 1,2,3,4 with their explanations above their respective code.
the conclusions are these: The instalment tends to repeat in events of underpayment / partial payment, for change in the version of the schedule, so the number can repeat even with 
full payment of the instalment if the schedule changes (NUM_INSTALMENT_VERSION). Beyond these dominant patterns we also found one more case where the instalment number 
can repeat and is highly correlated with what appears to be advance payment behavior. If you try to look for cases where "AMT_PAYMENT" == "AMT_INSTALMENT" and the schedule
don't change (NUM_INSTALMENT_VERSION) you will discover that there are no cases that meet these conditions at the same time (3rd analysis). But if you add that the payment can be 0 
("AMT_PAYMENT" == 0) it becomes possible to observe cases of full payment without reschedule. (4th analysis)"""

installments_payment_df["NEXT_INSTALLMENT_VERSION"] = installments_payment_df.groupby("SK_ID_PREV")["NUM_INSTALMENT_VERSION"].shift(-1)
same_version_versions_mask= (installments_payment_df ["NUM_INSTALMENT_VERSION"]==installments_payment_df["NEXT_INSTALLMENT_VERSION"])
full_installment_payment_mask= (installments_payment_df ["AMT_INSTALMENT"] == installments_payment_df["AMT_PAYMENT"])



#1- when the instalment number repeat but the installment was totaly paid
rows_repeated_installment_with_no_underpayment= analyze_repeated_instalments(full_installment_payment_mask, "of cases that repeat number of installment but it's not underpayment")

#2 - when the instalment number repeat and also the version
rows_same_version_and_installment_number= analyze_repeated_instalments(same_version_versions_mask,"of cases that repeat number of installment and version of the next row")

#3 - when the instalment number repeat, but it's not underpayment and the instalment version don't change (0 rows found, this does not happen in the dataset)
same_version_and_full_payment_mask = (same_version_versions_mask) & (full_installment_payment_mask)
rows_same_version_and_full_payment= analyze_repeated_instalments(same_version_and_full_payment_mask,"of cases that repeat number of installment and version of the next row")


#4 - the same that before but we allows to "AMT_INSTALMENT" being == 0. We want to include this because in previous analysis we discover that sometimes the number is repeated because 
#the installment was payment in advance and take another register repeating the instalment but with AMT_INSTALMENT being 0. (already paid)
full_payment_with_extra_row= (full_installment_payment_mask | (installments_payment_df["AMT_PAYMENT"] == 0))
same_version_and_full_payment_with_extra_row_mask = (same_version_versions_mask) & (full_payment_with_extra_row)
payed_in_advance_row= analyze_repeated_instalments(same_version_and_full_payment_with_extra_row_mask,"of cases that repeat instalment number for payment in advance")
dtale.show(get_full_sorted_serie_rows(payed_in_advance_row))



installments_payment_df.drop(
    columns=["NEXT_INSTALLMENT_VERSION"],
    inplace=True
)


we have 372 of cases that repeat number of installment but it's not underpayment

we have 651557 of cases that repeat number of installment and version of the next row

we have 0 of cases that repeat number of installment and version of the next row

we have 1427 of cases that repeat instalment number for payment in advance



In [ ]:
dtale.show(get_full_sorted_serie_rows(payed_in_advance_row))


2026-06-05 18:49:57,426 - INFO     - Executing shutdown due to inactivity...
2026-06-05 18:50:21,991 - INFO     - Executing shutdown...
2026-06-05 18:50:21,993 - INFO     - Not running with the Werkzeug Server, exiting by searching gc for BaseWSGIServer
2026-06-05 19:55:55,779 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c: